# Lab 2.4 &mdash; Branch, Score, Prune &mdash; and When to Stop Reflecting

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 1 &middot; Module 2 &mdash; Agentic Planning &amp; Reasoning**

### What you'll do
- Generate several candidate actions concurrently with <code>.batch()</code>
- Score them with a structured judge instead of reading them yourself
- Prune, and look at what you threw away
- Build a reflection loop that stops when it stops improving

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **Builds on Lab 2.1.** Same chain machinery, used two ways that both trade extra
> calls for a better answer &mdash; when they work.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-2-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 2 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- the two tools, carried through Module 2
from langchain_core.tools import tool

@tool
def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1003'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    """
    rec = LEDGER.get(ref)
    return json.dumps({"ref": ref, **rec}) if rec else f"no payment found with reference {ref!r}"


@tool
def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code such as 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {t.name: t for t in (lookup_payment, policy_for)}
print("tools:", list(TOOLS))

## Concept

**Tree-of-Thought** explores several routes, scores them, and keeps the promising ones.
**Reflection** drafts, criticises and revises. Both buy quality with extra calls, and both have a
point past which the extra calls buy nothing.

The framework parts that matter here:

- `chain.batch([...])` runs the branches **concurrently**, so width costs latency once, not
  *n* times;
- `with_structured_output(Score)` makes the judge return numbers you can sort, instead of a
  paragraph you have to read.

A scorer that returns prose is not a scorer. It is a second opinion you still have to interpret.

## Section 1 &mdash; Generate the branches, concurrently

One prompt, *n* different framings, one batched call.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

ANGLES = [
    "Answer strictly from the policy text, quoting its operative words.",
    "Answer as the operations desk: what do we physically do next, and who do we tell?",
    "Answer as the control function: what must NOT happen, and who owns the decision?",
]

def branch_chain():
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a payments operations analyst. {angle} Answer in two sentences."),
        ("human", "PAYMENT: {payment}\nPOLICY: {policy}\n\nWhat must happen next?"),
    ])
    return prompt | get_llm() | StrOutputParser()


def branches(ref: str) -> list[str]:
    """One candidate answer per angle, generated concurrently."""
    rec = LEDGER[ref]
    base = {"payment": json.dumps({"ref": ref, **rec}),
            "policy": POLICY.get(rec["reason_code"], "no policy applies")}
    return branch_chain().batch(BLANK)    # TODO: one input dict per angle

In [ ]:
# --- Self-check: Section 1   (input construction only -- no model call)
def _inputs():
    rec = LEDGER["PMT-1003"]
    base = {"payment": json.dumps({"ref": "PMT-1003", **rec}),
            "policy": POLICY[rec["reason_code"]]}
    return [{**base, "angle": a} for a in ANGLES]

check("one input per angle",         lambda: len(_inputs()) == len(ANGLES))
check("each input carries all three variables",
      lambda: all(set(d) == {"payment", "policy", "angle"} for d in _inputs()))
check("the angles really differ",    lambda: len({d["angle"] for d in _inputs()}) == len(ANGLES),
      "three copies of one prompt is not a tree")
check("the case data is shared across branches",
      lambda: len({d["payment"] for d in _inputs()}) == 1,
      "branches must differ in approach only, or you are comparing different questions")

## Section 2 &mdash; The scorer is the design

Whatever the judge rewards is what the tree will select for. Declare it as a schema so the
scores come back sortable.

In [ ]:
class Score(BaseModel):
    """A judge's verdict on one candidate answer."""
    grounded: int = Field(description="BLANK", ge=0, le=5)  # TODO: what is this scoring, 0 to 5?
    actionable: int = Field(description="0-5: is there a concrete next action a person could take?",
                            ge=0, le=5)
    safe: int = Field(description="0-5: does it respect who is allowed to decide? 0 if it "
                                  "proposes acting where policy reserves the decision for a human",
                      ge=0, le=5)
    why: str = Field(description="One short sentence justifying the lowest of the three scores")

    @property
    def total(self) -> int:
        return self.grounded + self.actionable + self.safe


def judge(ref: str, candidate: str) -> Score:
    rec = LEDGER[ref]
    return get_llm().with_structured_output(Score).invoke(
        "Score this proposed action against the policy. Be strict.\n"
        f"PAYMENT: {json.dumps({'ref': ref, **rec})}\n"
        f"POLICY: {POLICY.get(rec['reason_code'], 'no policy applies')}\n"
        f"CANDIDATE: {candidate}")

In [ ]:
# --- Self-check: Section 2   (schema + arithmetic -- no model call)
def _grounded_desc():
    d = Score.model_fields["grounded"].description
    if not d or d == "BLANK":
        raise NameError("Score.grounded still has no description")
    return d

check("the judge returns three numbers and a reason",
      lambda: set(Score.model_fields) == {"grounded", "actionable", "safe", "why"})
check("grounded says what it measures",
      lambda: len(_grounded_desc()) > 40)
check("the range is declared to the model",
      lambda: "0" in _grounded_desc() and "5" in _grounded_desc(),
      "a scale the model has to guess is a scale you cannot compare across runs")
def _rejects_out_of_range():
    try:
        Score(grounded=9, actionable=1, safe=1, why="x")
        return False
    except Exception:
        return True

check("out-of-range scores are rejected by the schema",
      lambda: _rejects_out_of_range())
check("total adds the three",
      lambda: Score(grounded=5, actionable=4, safe=3, why="x").total == 12)

## Section 3 &mdash; Prune, and look at what you lost

Keeping the top *k* is easy. Looking at what you dropped is the part people skip, and it is where
you find out your scorer is rewarding the wrong thing.

In [ ]:
def prune(candidates: list[str], scored: list[Score], keep: int = 1):
    """Return (kept, dropped) as lists of (candidate, score), best first."""
    pairs = sorted(zip(candidates, scored), key=lambda p: BLANK, reverse=True)  # TODO: by what?
    return pairs[:keep], pairs[keep:]

In [ ]:
# --- Self-check: Section 3
_cands = ["a", "b", "c"]
_scored = [Score(grounded=2, actionable=2, safe=2, why="x"),     # 6
           Score(grounded=5, actionable=5, safe=5, why="x"),     # 15
           Score(grounded=4, actionable=4, safe=4, why="x")]     # 12

check("the best candidate is kept",     lambda: prune(_cands, _scored)[0][0][0] == "b")
check("the rest are returned, not lost", lambda: len(prune(_cands, _scored)[1]) == 2)
check("the dropped list is also ordered",
      lambda: [c for c, _ in prune(_cands, _scored)[1]] == ["c", "a"],
      "you cannot review what you pruned if it comes back shuffled")
check("keep=2 keeps two",               lambda: len(prune(_cands, _scored, keep=2)[0]) == 2)

## Section 4 &mdash; Reflection, and the knee

Reflection improves a draft by criticising it. The gain per round drops fast &mdash; usually a real
improvement on round one, a small one on round two, and noise after that. Stop when the critic
stops finding anything.

In [ ]:
CRITIC = ("You are a strict reviewer. List the specific faults in this proposed action against "
          "the policy: invented facts, missing next step, or acting where a human must decide. "
          "If there is nothing material to fix, reply with exactly: NO ISSUES")

def critic_is_done(critique: str) -> bool:
    """Has reflecting stopped paying? True when the critic found nothing material."""
    return BLANK                      # TODO: how does the critic say "nothing to fix"?


def reflect(ref: str, draft: str, rounds: int = 3) -> dict:
    """Draft -> critique -> revise, stopping when the critic has nothing left."""
    rec = LEDGER[ref]
    context = (f"PAYMENT: {json.dumps({'ref': ref, **rec})}\n"
               f"POLICY: {POLICY.get(rec['reason_code'], 'no policy applies')}")
    history = [("draft", draft)]
    for i in range(rounds):
        critique = ask(f"{context}\n\nPROPOSED ACTION: {draft}", system=CRITIC)
        history.append((f"critique {i+1}", critique))
        if critic_is_done(critique):
            history.append(("stopped", f"critic found nothing on round {i+1}"))
            break
        draft = ask(f"{context}\n\nPROPOSED ACTION: {draft}\n\nFAULTS FOUND: {critique}\n\n"
                    "Rewrite the action in two sentences, fixing only what was faulted.")
        history.append((f"revision {i+1}", draft))
    return {"final": draft, "history": history}

In [ ]:
# --- Self-check: Section 4   (the stop rule, on canned critiques -- no model call)
check("the critic is told how to say 'nothing to fix'",
      lambda: "NO ISSUES" in CRITIC,
      "a critic with no way to pass will always invent a fault, and you will loop forever")
check("an all-clear critique stops the loop",
      lambda: critic_is_done("NO ISSUES") is True)
check("a critique with faults does not stop it",
      lambda: critic_is_done("It proposes releasing a sanctions hold.") is False)
check("the check is case-insensitive",
      lambda: critic_is_done("no issues") is True,
      "models do not honour your capitalisation")
check("an empty critique does not stop it",
      lambda: critic_is_done("") is False,
      "a model that returned nothing has not told you the draft is good")

## Run it for real &mdash; the tree

In [ ]:
if llm_ready():
    def _tree():
        ref = "PMT-1005"          # a sanctions hold: the safe score is the one that separates them
        cands = branches(ref)
        print(f"{len(cands)} branches generated concurrently\n")
        scored = [judge(ref, c) for c in cands]
        kept, dropped = prune(cands, scored, keep=1)

        for label, group in (("KEPT", kept), ("dropped", dropped)):
            for cand, sc in group:
                print(f"[{label:7}] total={sc.total:2}  g={sc.grounded} a={sc.actionable} s={sc.safe}")
                print(f"          {cand.strip()[:200]}")
                print(f"          why: {sc.why[:160]}\n")
        return kept, dropped
    TREE = guard(_tree)

## Run it for real &mdash; the reflection loop

In [ ]:
if llm_ready():
    def _reflect():
        weak = "Release the payment once the counterparty confirms by email."
        out = reflect("PMT-1005", weak)
        for label, text in out["history"]:
            print(f"--- {label} ---")
            print(text.strip()[:400] + "\n")
        print("=== final ===")
        print(out["final"].strip()[:400])
    guard(_reflect)

### Read it

**The tree.** Read the `why` on the branch that lost. PMT-1005 is a sanctions hold, so any answer
proposing to release or cancel it should score 0 on `safe` no matter how fluent it is &mdash; and the
operations-desk angle is the one most likely to write exactly that. If the judge scored it well
anyway, your scorer is the problem, not the branch. Fix the scorer before you widen the tree.

**The reflection loop.** The deliberately weak draft proposes releasing a sanctions hold on an
email confirmation, which is precisely what the policy forbids. Watch round one demolish it and
round two do much less. That shape &mdash; a large first gain, then a knee &mdash; is what you should
expect, and it is why `rounds=3` with an early stop beats `rounds=10`.

And note the failure mode built into the stop rule: a critic with no way to say "nothing to fix"
will always find something, because that is what you asked it for. `NO ISSUES` is not a nicety.

In [ ]:
score()

## Your turn

1. Add a fourth angle that is deliberately reckless &mdash; "answer as the client relationship
   manager who wants this paid today" &mdash; and confirm the judge scores it 0 on `safe`. If it does
   not, tighten the field description until it does.
2. Score all three branches in **one** batched call instead of three serial ones. You will need
   `with_structured_output(Score).batch(...)`. Measure the wall-clock difference.
3. Run `reflect` on an answer that is already correct. How many rounds before the critic starts
   inventing faults? That number is your real ceiling on reflection.